In [1]:
# =============================================================================
# Para instalar as dependências, rode no terminal:
#   pip install pandas numpy scikit-learn pyarrow
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pickle
import os

In [2]:
# =============================================================================
# ETAPA 1 — INGESTÃO E VALIDAÇÃO INICIAL
# Carregamos o arquivo CSV e exibimos informações básicas para entender os dados
# =============================================================================
print("=" * 60)
print("ETAPA 1 — Ingestão e validação inicial")
print("=" * 60)

df = pd.read_csv("used_car_price_prediction_1M.csv")

print(f"Linhas: {len(df):,}")
print(f"Colunas: {df.shape[1]}")
print("\nTipos de dados:")
print(df.dtypes)
print("\nValores faltantes:")
missing = df.isnull().sum()
print(missing[missing > 0])
print("\nEstatísticas do target (Price):")
print(df["Price"].describe().apply(lambda x: f"{x:,.2f}"))

ETAPA 1 — Ingestão e validação inicial
Linhas: 1,005,000
Colunas: 20

Tipos de dados:
Brand                   str
Model                   str
Year                  int64
Mileage_kmpl        float64
Engine_CC           float64
Horsepower          float64
Fuel_Type               str
Transmission            str
Owner_Type              str
Color                   str
City                    str
Kms_Driven            int64
Insurance_Valid       int64
Service_History       int64
Accidents             int64
Tax_Paid              int64
Number_of_Doors       int64
Seats                 int64
Registration_Age      int64
Price               float64
dtype: object

Valores faltantes:
Mileage_kmpl    80390
Engine_CC       80403
Horsepower      80403
Fuel_Type       78733
Transmission    80385
Color           80419
City            80430
dtype: int64

Estatísticas do target (Price):
count     1,005,000.00
mean      1,012,896.96
std       1,141,644.94
min          50,000.00
25%         343,350.41
50%  

In [3]:
# =============================================================================
# ETAPA 2 — REMOÇÃO DE DUPLICATAS
# Registros idênticos atrapalham o modelo (ele "aprende" o mesmo caso duas vezes)
# =============================================================================
print("\n" + "=" * 60)
print("ETAPA 2 — Remoção de duplicatas")
print("=" * 60)

antes = len(df)
df = df.drop_duplicates()
depois = len(df)
print(f"Duplicatas removidas: {antes - depois:,}")
print(f"Registros restantes:  {depois:,}")


ETAPA 2 — Remoção de duplicatas
Duplicatas removidas: 5,000
Registros restantes:  1,000,000


In [ ]:
# =============================================================================
# ETAPA 3 — TRATAMENTO DE VALORES FALTANTES
# + ou - 8% dos registros têm valores ausentes em 7 colunas.
# Estratégia: mediana para números (resistente a outliers),
# 'Unknown' para textos (preserva a informação de que estava vazio)
# =============================================================================
print("\n" + "=" * 60)
print("ETAPA 3 — Tratamento de valores faltantes")
print("=" * 60)

# Numéricas: preencher com a mediana
numeric_cols = ["Mileage_kmpl", "Engine_CC", "Horsepower"]
for col in numeric_cols:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)
    print(f"{col}: mediana = {mediana:.2f}")

# Categóricas: preencher com 'Unknown'
categorical_missing = ["Fuel_Type", "Transmission", "Color", "City"]
for col in categorical_missing:
    df[col] = df[col].fillna("Unknown")
    print(f"{col}: preenchido com 'Unknown'")

print(f"\nValores faltantes restantes: {df.isnull().sum().sum()}")


ETAPA 3 — Tratamento de valores faltantes
Mileage_kmpl: mediana = 18.00
Engine_CC: mediana = 1500.20
Horsepower: mediana = 120.06
Fuel_Type: preenchido com 'Unknown'
Transmission: preenchido com 'Unknown'
Color: preenchido com 'Unknown'
City: preenchido com 'Unknown'

Valores faltantes restantes: 0


In [5]:
# =============================================================================
# ETAPA 4 — TRATAMENTO DE OUTLIERS E TRANSFORMAÇÃO DO TARGET
# O preço vai de 50 mil a 37,8 milhões — distribuição extremamente assimétrica.
# Solução: transformação logarítmica (log(Price)) que comprime a escala.
# O modelo aprende na escala log; para converter de volta: np.expm1(predição)
# =============================================================================
print("\n" + "=" * 60)
print("ETAPA 4 — Transformação do target (Price)")
print("=" * 60)

Q1  = df["Price"].quantile(0.25)
Q3  = df["Price"].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 3 * IQR
outliers = (df["Price"] > limite_superior).sum()
print(f"Outliers extremos detectados (acima de {limite_superior:,.0f}): {outliers:,} ({outliers/len(df)*100:.2f}%)")
print("→ Optamos por manter e usar transformação logarítmica")

df["Price_log"] = np.log1p(df["Price"])
print(f"\nPrice_log — média: {df['Price_log'].mean():.4f}, std: {df['Price_log'].std():.4f}")
print("(distribuição muito mais simétrica agora)")


ETAPA 4 — Transformação do target (Price)
Outliers extremos detectados (acima de 4,144,520): 5,803 (0.58%)
→ Optamos por manter e usar transformação logarítmica

Price_log — média: 13.2783, std: 1.2086
(distribuição muito mais simétrica agora)


In [6]:
# =============================================================================
# ETAPA 5 — PADRONIZAÇÃO DAS VARIÁVEIS CATEGÓRICAS
# O dataset tem erros intencionais: espaços extras, capitalização errada, typos.
# Exemplos encontrados: 'PETROL', ' Diesel', 'hybridd', 'electrik'
# =============================================================================
print("\n" + "=" * 60)
print("ETAPA 5 — Padronização de variáveis categóricas")
print("=" * 60)

# Padronizar todas as colunas de texto
text_cols = ["Brand", "Model", "Fuel_Type", "Transmission", "Owner_Type", "Color", "City"]
for col in text_cols:
    df[col] = df[col].str.strip().str.title()

# Corrigir erros tipográficos identificados na análise
correcoes = {
    "Fuel_Type": {
        "Hybridd":  "Hybrid",
        "Electrik": "Electric",
        "Cng":      "CNG",
    }
}
for col, mapa in correcoes.items():
    df[col] = df[col].replace(mapa)
    print(f"{col}: {len(mapa)} correções aplicadas → {sorted(df[col].unique())}")


ETAPA 5 — Padronização de variáveis categóricas
Fuel_Type: 3 correções aplicadas → ['CNG', 'Diesel', 'Electric', 'Hybrid', 'Petrol', 'Unknown']


In [7]:
# =============================================================================
# ETAPA 6 — FEATURE ENGINEERING (Engenharia de Features)
# Criamos novas variáveis que podem ajudar o modelo a aprender melhor.
# Cada nova feature captura uma relação que não estava explícita nos dados.
# =============================================================================
print("\n" + "=" * 60)
print("ETAPA 6 — Feature Engineering")
print("=" * 60)

# Idade real do veículo (mais intuitivo que Registration_Age)
df["Vehicle_Age"] = 2025 - df["Year"]
print("✓ Vehicle_Age: idade do veículo em anos")

# Indicador de marca premium (veículos de luxo têm precificação diferente)
marcas_premium = ["Bmw", "Mercedes", "Audi", "Porsche", "Jaguar", "Land Rover", "Volvo", "Lexus"]
df["Is_Premium"] = df["Brand"].isin(marcas_premium).astype(int)
print(f"✓ Is_Premium: {df['Is_Premium'].sum():,} veículos premium ({df['Is_Premium'].mean()*100:.1f}%)")

# Potência por cilindrada: mede eficiência mecânica do motor
df["Power_per_CC"] = df["Horsepower"] / (df["Engine_CC"] + 1)
print(f"✓ Power_per_CC: média = {df['Power_per_CC'].mean():.4f}")

# Quilometragem por ano: uso anual médio (quilômetro/ano)
df["Kms_per_Year"] = df["Kms_Driven"] / (df["Vehicle_Age"] + 1)
print(f"✓ Kms_per_Year: média = {df['Kms_per_Year'].mean():,.0f} km/ano")

# Score de confiabilidade: combina seguros, histórico e acidentes
df["Reliability_Score"] = (
    df["Insurance_Valid"] +
    df["Service_History"] +
    df["Tax_Paid"]        -
    df["Accidents"]
)
print(f"✓ Reliability_Score: varia de {df['Reliability_Score'].min()} a {df['Reliability_Score'].max()}")


ETAPA 6 — Feature Engineering
✓ Vehicle_Age: idade do veículo em anos
✓ Is_Premium: 231,044 veículos premium (23.1%)
✓ Power_per_CC: média = 0.0804
✓ Kms_per_Year: média = 13,819 km/ano
✓ Reliability_Score: varia de -4 a 3


In [8]:
# =============================================================================
# ETAPA 7 — ENCODING DAS VARIÁVEIS CATEGÓRICAS
# Algoritmos de ML só entendem números. Precisamos converter texto em números.
#
# Duas estratégias:
# - One-Hot Encoding (OHE): para colunas com poucas categorias (ex: Fuel_Type)
#   → cria uma coluna binária (0/1) para cada categoria
# - Label Encoding: para colunas com muitas categorias (ex: Brand, Model)
#   → substitui cada texto por um número inteiro
# =============================================================================
print("\n" + "=" * 60)
print("ETAPA 7 — Encoding de variáveis categóricas")
print("=" * 60)

# One-Hot Encoding: baixa cardinalidade
ohe_cols = ["Fuel_Type", "Transmission", "Owner_Type"]
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)
print(f"✓ OHE aplicado em: {ohe_cols}")

# Label Encoding: alta cardinalidade
label_encoders = {}
le_cols = ["Brand", "Model", "City", "Color"]
for col in le_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    df = df.drop(columns=[col])
print(f"✓ Label Encoding aplicado em: {le_cols}")

# Remover colunas redundantes ou desnecessárias
# Year → substituído por Vehicle_Age
# Registration_Age → redundante com Vehicle_Age
# Price → usaremos Price_log como target
df = df.drop(columns=["Year", "Registration_Age", "Price"])
print(f"\nShape final: {df.shape}")
print(f"Colunas: {list(df.columns)}")



ETAPA 7 — Encoding de variáveis categóricas
✓ OHE aplicado em: ['Fuel_Type', 'Transmission', 'Owner_Type']
✓ Label Encoding aplicado em: ['Brand', 'Model', 'City', 'Color']

Shape final: (1000000, 30)
Colunas: ['Mileage_kmpl', 'Engine_CC', 'Horsepower', 'Kms_Driven', 'Insurance_Valid', 'Service_History', 'Accidents', 'Tax_Paid', 'Number_of_Doors', 'Seats', 'Price_log', 'Vehicle_Age', 'Is_Premium', 'Power_per_CC', 'Kms_per_Year', 'Reliability_Score', 'Fuel_Type_Diesel', 'Fuel_Type_Electric', 'Fuel_Type_Hybrid', 'Fuel_Type_Petrol', 'Fuel_Type_Unknown', 'Transmission_Manual', 'Transmission_Unknown', 'Owner_Type_Fourth+', 'Owner_Type_Second', 'Owner_Type_Third', 'Brand_enc', 'Model_enc', 'City_enc', 'Color_enc']


In [9]:
# =============================================================================
# ETAPA 8 — DIVISÃO TREINO / TESTE
# Dividimos os dados em 80% para treino e 20% para teste.
# O modelo aprende com o treino e é avaliado no teste (dados que nunca viu).
# random_state=42 garante que a divisão seja sempre a mesma (reprodutibilidade)
# =============================================================================
print("\n" + "=" * 60)
print("ETAPA 8 — Divisão treino / teste")
print("=" * 60)

X = df.drop(columns=["Price_log"])   # features (entradas)
y = df["Price_log"]                  # target (saída)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Treino: {len(X_train):,} registros (80%)")
print(f"Teste:  {len(X_test):,} registros (20%)")
print(f"Features: {X.shape[1]}")
print(f"\nMédia do target — Treino: {y_train.mean():.4f} | Teste: {y_test.mean():.4f}")
print("(médias similares = divisão bem distribuída ✓)")



ETAPA 8 — Divisão treino / teste
Treino: 800,000 registros (80%)
Teste:  200,000 registros (20%)
Features: 29

Média do target — Treino: 13.2787 | Teste: 13.2768
(médias similares = divisão bem distribuída ✓)


In [11]:
# =============================================================================
# ETAPA 9 — EXPORTAR DADOS PROCESSADOS
# Salvamos tudo para que o colega possa continuar com ML e Visualização.
# =============================================================================
print("\n" + "=" * 60)
print("ETAPA 9 — Exportando arquivos")
print("=" * 60)

# Dataset completo processado (formato Parquet = mais eficiente que CSV)
df.to_parquet("data_processed.parquet", index=False)
print(f"✓ data_processed.parquet ({os.path.getsize('data_processed.parquet')/1024**2:.1f} MB)")

# Splits prontos para o modelo
with open("train_test_split.pkl", "wb") as f:
    pickle.dump((X_train, X_test, y_train, y_test), f)
print(f"✓ train_test_split.pkl ({os.path.getsize('train_test_split.pkl')/1024**2:.1f} MB)")


ETAPA 9 — Exportando arquivos
✓ data_processed.parquet (53.0 MB)
✓ train_test_split.pkl (185.0 MB)


In [13]:
# =============================================================================
# RESUMO FINAL
# =============================================================================
print("\n" + "=" * 60)
print("RESUMO FINAL DO ETL")
print("=" * 60)
print(f"Registros originais:     1.005.000")
print(f"Após remoção de duplic.: 1.000.000")
print(f"Features geradas:        {X.shape[1]}")
print(f"  → 15 numéricas originais")
print(f"  → 5 novas features (feature engineering)")
print(f"  → 9 colunas OHE (Fuel_Type, Transmission, Owner_Type)")
print(f"  → 4 colunas Label Encoded (Brand, Model, City, Color)")
print(f"\nTarget: Price_log")
print(f"  → Para converter predições de volta: np.expm1(y_pred)")
print(f"\nArquivos para o Lucas colocar no ML:")
print(f"  1. data_processed.parquet  → dataset completo limpo")
print(f"  2. train_test_split.pkl    → splits X_train, X_test, y_train, y_test")
print("\nETL concluído com sucesso! ✓")


RESUMO FINAL DO ETL
Registros originais:     1.005.000
Após remoção de duplic.: 1.000.000
Features geradas:        29
  → 15 numéricas originais
  → 5 novas features (feature engineering)
  → 9 colunas OHE (Fuel_Type, Transmission, Owner_Type)
  → 4 colunas Label Encoded (Brand, Model, City, Color)

Target: Price_log
  → Para converter predições de volta: np.expm1(y_pred)

Arquivos para o Lucas colocar no ML:
  1. data_processed.parquet  → dataset completo limpo
  2. train_test_split.pkl    → splits X_train, X_test, y_train, y_test

ETL concluído com sucesso! ✓
